In [1]:
import torch
from torch import nn

net=nn.Sequential(
    nn.Linear(4,8),
    nn.ReLU(),
    nn.Linear(8,1)
)
x=torch.rand(size=(5,4))
net(x)

tensor([[0.2770],
        [0.2543],
        [0.3117],
        [0.2876],
        [0.2495]], grad_fn=<AddmmBackward0>)

In [2]:
print(net[2].state_dict())

OrderedDict([('weight', tensor([[-0.0682, -0.2706, -0.1496,  0.1865, -0.0730,  0.2550, -0.2312, -0.3299]])), ('bias', tensor([0.3020]))])


In [6]:
net[2].bias.data=torch.tensor([0.0])

In [7]:
print(net[2].bias.data)

tensor([0.])


In [9]:
print(net[2].bias.grad)

None


In [11]:
for name,param in net[0].named_parameters():
    print(name,param.shape,param)

weight torch.Size([8, 4]) Parameter containing:
tensor([[-0.4560, -0.1967,  0.3039, -0.2368],
        [-0.2600,  0.0980, -0.1924,  0.0320],
        [-0.3259, -0.2692,  0.2964,  0.3860],
        [-0.1421, -0.0296,  0.1312, -0.2105],
        [ 0.0881, -0.3619, -0.3132, -0.3941],
        [ 0.3883, -0.4759,  0.3438, -0.3444],
        [ 0.4780, -0.0923, -0.2337,  0.0185],
        [ 0.0632, -0.4158,  0.3162,  0.0620]], requires_grad=True)
bias torch.Size([8]) Parameter containing:
tensor([ 0.3503, -0.0580, -0.1514, -0.0883, -0.1969,  0.0268,  0.0230, -0.3578],
       requires_grad=True)


In [12]:
for name,param in net.named_parameters():
    print(name,param.shape,param)

0.weight torch.Size([8, 4]) Parameter containing:
tensor([[-0.4560, -0.1967,  0.3039, -0.2368],
        [-0.2600,  0.0980, -0.1924,  0.0320],
        [-0.3259, -0.2692,  0.2964,  0.3860],
        [-0.1421, -0.0296,  0.1312, -0.2105],
        [ 0.0881, -0.3619, -0.3132, -0.3941],
        [ 0.3883, -0.4759,  0.3438, -0.3444],
        [ 0.4780, -0.0923, -0.2337,  0.0185],
        [ 0.0632, -0.4158,  0.3162,  0.0620]], requires_grad=True)
0.bias torch.Size([8]) Parameter containing:
tensor([ 0.3503, -0.0580, -0.1514, -0.0883, -0.1969,  0.0268,  0.0230, -0.3578],
       requires_grad=True)
2.weight torch.Size([1, 8]) Parameter containing:
tensor([[-0.0682, -0.2706, -0.1496,  0.1865, -0.0730,  0.2550, -0.2312, -0.3299]],
       requires_grad=True)
2.bias torch.Size([1]) Parameter containing:
tensor([0.], requires_grad=True)


In [15]:
def block1():
    return nn.Sequential(
        nn.Linear(4,8),
        nn.ReLU(),
        nn.Linear(8,4),
        nn.ReLU()
    )

def block2():
    net=nn.Sequential()
    for i in range(4):
        net.add_module(f'block{i}',block1())
    return net
rgnet=nn.Sequential(block2(),nn.Linear(4,1))
rgnet(x)

tensor([[0.4410],
        [0.4410],
        [0.4410],
        [0.4410],
        [0.4410]], grad_fn=<AddmmBackward0>)

In [16]:
print(rgnet)

Sequential(
  (0): Sequential(
    (block0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


In [23]:
def init_normal(m):
    if type(m)==nn.Linear:
        nn.init.normal_(m.weight,mean=0,std=0.01)
        nn.init.zeros_(m.bias)
        #m.weight=nn.init.normal_(mean=0,std=0.01)

net.apply(init_normal)
net[0].weight.data[0],net[0].bias.data

(tensor([-0.0125,  0.0003,  0.0114, -0.0023]),
 tensor([0., 0., 0., 0., 0., 0., 0., 0.]))

In [24]:
def init_constant(m):
    if type(m)==nn.Linear:
        nn.init.constant_(m.weight,1)
        nn.init.zeros_(m.bias)

net.apply(init_constant)
net[0].weight.data[0],net[0].bias.data

(tensor([1., 1., 1., 1.]), tensor([0., 0., 0., 0., 0., 0., 0., 0.]))

In [27]:
def xavier(m):
    if type(m)==nn.Linear:
        nn.init.xavier_uniform_(m.weight)

def init_42(m):
    if type(m)==nn.Linear:
        nn.init.constant_(m.weight,42)

net[0].apply(xavier)
net[2].apply(init_42)
print(net[0].weight.data[0])
print(net[2].weight.data)

tensor([-0.3995,  0.2825,  0.3842,  0.6468])
tensor([[42., 42., 42., 42., 42., 42., 42., 42.]])


In [28]:
net[0].weight.data[:]+=1
net[0].weight.data[0,0]=42
net[0].weight.data

tensor([[42.0000,  1.2825,  1.3842,  1.6468],
        [ 1.1069,  0.9650,  0.6008,  0.4220],
        [ 0.8044,  0.8431,  1.1034,  0.4926],
        [ 0.8633,  1.0757,  1.5777,  1.5216],
        [ 1.6726,  1.5485,  0.9289,  0.4299],
        [ 0.7647,  0.7458,  0.8640,  1.0274],
        [ 1.0077,  0.9502,  0.4283,  0.6045],
        [ 1.6076,  0.4260,  0.8562,  0.4830]])

In [30]:
shared=nn.Linear(8,8)
net=nn.Sequential(
    nn.Linear(4,8),
    nn.ReLU(),
    shared,
    nn.ReLU(),
    shared,
    nn.Linear(8,74),
    nn.ReLU()
)
net(x)
print(net[2].weight.data==net[4].weight.data)
net[2].weight.data[0,0]=42
print(net[2].weight.data==net[4].weight.data)

tensor([[True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True]])
tensor([[True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True]])
